In [ ]:
import os
import subprocess
import sys


print("=" * 60)
print("NEMOTRON-3 NANO 30B-A3B - LoRA SFT Training (v21)")
print("=" * 60)

# ──────────────────────────────────────────────────────────
# 1. Blackwell Environment Setup (Kaggle G4 Handshake)
# ──────────────────────────────────────────────────────────
print("\n[1/7] Setting up Blackwell environment...")
UTILITY_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"
if os.path.exists(UTILITY_PATH):
    subprocess.run(f"tar -cf - -C {UTILITY_PATH} . | tar -xf - -C /tmp", shell=True, check=True)
    for binary in ["ptxas", "ptxas-blackwell"]:
        bin_path = f"/tmp/triton/backends/nvidia/bin/{binary}"
        if os.path.exists(bin_path):
            subprocess.run(f"chmod +x {bin_path}", shell=True, check=True)
            print(f"  Set execution permission on {binary}")
    sys.path.insert(0, "/tmp")
    print("  Blackwell utility script initialized in /tmp")
else:
    print(f"  WARNING: Utility script not found at {UTILITY_PATH}")

# *** THE FIX: Set TRITON_PTXAS_PATH for Blackwell ***
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
print(f"  TRITON_PTXAS_PATH = {os.environ['TRITON_PTXAS_PATH']}")


In [ ]:
# ──────────────────────────────────────────────────────────
# 2. Verify hardware & dependencies
# ──────────────────────────────────────────────────────────
print("\n[2/7] Verifying hardware and dependencies...")
import torch


print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {prop.name} ({prop.total_mem / 1024**3:.1f} GB)")

# Check mandatory dependencies
for pkg_name in ["mamba_ssm", "causal_conv1d", "peft", "bitsandbytes", "cutlass"]:
    try:
        __import__(pkg_name)
        print(f"  {pkg_name}: OK")
    except ImportError:
        print(f"  {pkg_name}: MISSING - installing...")
        install_name = "nvidia-cutlass" if pkg_name == "cutlass" else pkg_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_name])


In [ ]:
# ──────────────────────────────────────────────────────────
# 3. Load Model + Tokenizer + LoRA
# ──────────────────────────────────────────────────────────
print("\n[3/7] Loading model and tokenizer...")

import kagglehub
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer


model_path = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"  Model path: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  Tokenizer loaded. Vocab size: {len(tokenizer)}")

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print(f"  Model loaded on {next(model.parameters()).device}")

# LoRA config: r=32, targeting MoE in/out projections
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["in_proj", "out_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Quick forward pass sanity check
dummy = torch.randint(0, model.config.vocab_size, (1, 8)).to(next(model.parameters()).device)
with torch.no_grad():
    out = model(dummy)
print(f"  Forward pass OK. Logits shape: {out.logits.shape}")


In [ ]:
# ──────────────────────────────────────────────────────────
# 4. Load and Format Training Data
# ──────────────────────────────────────────────────────────
print("\n[4/7] Loading training data...")

import pandas as pd


TRAIN_PATH = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_PATH)
print(f"  Loaded {len(df)} rows from train.csv")
print(f"  Columns: {list(df.columns)}")
print(f"  Sample row:\n{df.iloc[0].to_dict()}")

# Format each row as a chat conversation with thinking enabled.
# The competition expects reasoning, so we use enable_thinking=True style:
#   system prompt tells the model to think step-by-step,
#   user provides the question, assistant provides the answer.

def format_chat(row):
    """Format a training row as a tokenized chat with thinking."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Think through problems step by step "
                "inside <think>...</think> tags before giving your final answer."
            ),
        },
        {"role": "user", "content": str(row["question"])},
        {"role": "assistant", "content": str(row["answer"])},
    ]
    # apply_chat_template with enable_thinking if supported
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=True,
        )
    except TypeError:
        # Fallback if enable_thinking not supported by this tokenizer
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    return text

# Tokenize all examples
MAX_SEQ_LEN = 2048  # Conservative for memory; increase if VRAM allows

print("  Formatting and tokenizing...")
texts = [format_chat(row) for _, row in df.iterrows()]
print(f"  Sample formatted text (first 500 chars):\n{texts[0][:500]}")

# Tokenize
from torch.utils.data import DataLoader, Dataset


class SFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.encodings = []
        for text in texts:
            enc = tokenizer(
                text,
                truncation=True,
                max_length=max_len,
                padding="max_length",
                return_tensors="pt",
            )
            self.encodings.append({
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
            })

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]

train_dataset = SFTDataset(texts, tokenizer, MAX_SEQ_LEN)
print(f"  Dataset ready: {len(train_dataset)} examples, max_len={MAX_SEQ_LEN}")


In [ ]:
# ──────────────────────────────────────────────────────────
# 5. SFT Training Loop (1 Epoch)
# ──────────────────────────────────────────────────────────
print("\n[5/7] Starting SFT training...")


# Training hyperparameters (conservative for 1 epoch)
BATCH_SIZE = 1          # Nemotron 30B is large; gradient accumulation compensates
GRAD_ACCUM_STEPS = 8    # Effective batch size = 8
LEARNING_RATE = 2e-5
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

# Linear warmup then cosine decay
from torch.optim.lr_scheduler import OneCycleLR


total_steps = len(train_loader) // GRAD_ACCUM_STEPS
scheduler = OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=max(total_steps, 1),
    pct_start=min(WARMUP_STEPS / max(total_steps, 1), 0.1),
    anneal_strategy="cos",
)

model.train()
device = next(model.parameters()).device

running_loss = 0.0
log_interval = 50  # Print every 50 steps
global_step = 0

print(f"  Total examples: {len(train_dataset)}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} accum = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"  Total optimizer steps: {total_steps}")
print(f"  Learning rate: {LEARNING_RATE}")

optimizer.zero_grad()

for step, batch in enumerate(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    # Standard causal LM: labels = input_ids (shifted inside the model)
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=input_ids,
    )
    loss = outputs.loss / GRAD_ACCUM_STEPS
    loss.backward()

    running_loss += loss.item()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

        if global_step % log_interval == 0:
            avg_loss = running_loss / log_interval
            lr = scheduler.get_last_lr()[0]
            print(f"  Step {global_step}/{total_steps} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")
            running_loss = 0.0

# Handle remaining gradients
if (step + 1) % GRAD_ACCUM_STEPS != 0:
    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    optimizer.zero_grad()
    global_step += 1

print(f"  Training complete! Final step: {global_step}")


In [ ]:
# ──────────────────────────────────────────────────────────
# 6. Save LoRA Adapter
# ──────────────────────────────────────────────────────────
print("\n[6/7] Saving LoRA adapter...")

adapter_path = "nemotron_lora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"  Adapter saved to {adapter_path}/")

# List saved files
import glob


saved_files = glob.glob(f"{adapter_path}/*")
for f in sorted(saved_files):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f):40s} {size_mb:.2f} MB")


In [ ]:
# ──────────────────────────────────────────────────────────
# 7. Package Submission
# ──────────────────────────────────────────────────────────
print("\n[7/7] Packaging submission.zip...")

subprocess.run(
    f"cd {adapter_path} && zip -r ../submission.zip ./*",
    shell=True,
    check=True,
)

submission_size = os.path.getsize("submission.zip") / (1024 * 1024)
print(f"  submission.zip created: {submission_size:.2f} MB")

print("\n" + "=" * 60)
print("TRAINING COMPLETED SUCCESSFULLY")
print(f"  Model: Nemotron-3-Nano-30B-A3B")
print(f"  LoRA rank: 32")
print(f"  Training: 1 epoch SFT on competition train.csv")
print(f"  TRITON_PTXAS_PATH: {os.environ.get('TRITON_PTXAS_PATH', 'NOT SET')}")
print("=" * 60)
